In [ ]:
import os

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

from agent.helpers.docker import list_containers as list_containers_raw

load_dotenv()

In [ ]:
@tool
async def list_containers() -> str:
    """List all Docker containers with their current state and image."""
    containers = await list_containers_raw()
    lines = [
        f"{(c.get('Names') or ['?'])[0].lstrip('/')}: {c.get('State')} ({c.get('Image')})"
        for c in containers
    ]
    return "\n".join(lines) or "(no containers found)"

In [19]:
llm = ChatOpenAI(
    model=os.environ["LLM_MODEL"],
    base_url=os.environ["LLM_BASE_URL"],
    api_key=os.environ.get("LLM_API_KEY"),
    temperature=0,
).bind_tools([list_containers])

In [20]:
SYSTEM = "You are an infrastructure-operations assistant. Use the list_containers tool when relevant, then answer concisely."

transcript: list = []


async def ask(question: str) -> str:
    """Ask one question. Makes at most one list_containers call, then returns the model's answer."""
    messages = [SystemMessage(SYSTEM), HumanMessage(question)]
    reply = await llm.ainvoke(messages)
    messages.append(reply)
    if reply.tool_calls:
        call = reply.tool_calls[0]
        result = await list_containers.ainvoke(call["args"] or {})
        messages.append(
            ToolMessage(tool_call_id=call["id"], content=str(result)[:4000])
        )
        reply = await llm.ainvoke(messages)
        messages.append(reply)
    transcript[:] = messages
    return reply.content

In [21]:
print(await ask("Which containers exist right now and what state are they in?"))



Here are the containers currently existing and their states:

*   **marconi-sbx**: exited (image: ubuntu:24.04)
*   **bifrost-gateway-1**: exited (image: bifrost-gateway)


In [22]:
def show_transcript(transcript: list) -> None:
    """Print each message with its tool calls, token usage, and reasoning."""
    for m in transcript:
        label = type(m).__name__
        body = m.content if not isinstance(m, ToolMessage) else f"[tool] {m.content}"
        if not body:
            body = "(tool call only)" if getattr(m, "tool_calls", None) else "(empty)"
        print(f"{label}: {body.replace(chr(10), chr(10) + '  ')}")

        reasoning = m.additional_kwargs.get("reasoning_content")
        if reasoning:
            print(f"  reasoning: {reasoning}")

        for call in getattr(m, "tool_calls", None) or []:
            print(f"  tool_call: {call['name']}({call['args']})")

        usage = getattr(m, "usage_metadata", None)
        if usage:
            print(
                f"  tokens: in={usage['input_tokens']} out={usage['output_tokens']} total={usage['total_tokens']}"
            )

In [23]:
show_transcript(transcript)

SystemMessage: You are an infrastructure-operations assistant. Use the list_containers tool when relevant, then answer concisely.
HumanMessage: Which containers exist right now and what state are they in?
AIMessage: (tool call only)
  tool_call: list_containers({})
  tokens: in=294 out=142 total=436
ToolMessage: [tool] marconi-sbx: exited (ubuntu:24.04)
  bifrost-gateway-1: exited (bifrost-gateway)
AIMessage: 
  
  Here are the containers currently existing and their states:
  
  *   **marconi-sbx**: exited (image: ubuntu:24.04)
  *   **bifrost-gateway-1**: exited (image: bifrost-gateway)
  tokens: in=358 out=151 total=509
